## Convert ATMO brown dwarf isochrones from txt to hdf5

**Input**: Interpolated ATMO grids (by A. Vani)

**Output**: ATMO grids in dhf5

In [1]:
import os
import glob
import h5py
import numpy as np
from pathlib import Path


In [2]:
base_dir = 'from_Akash/ATMO'

models = ['isochrones_CEQ_MKO_WISE_IRAC',
          'isochrones_CEQ_NIRCAM_modA',
          'isochrones_NEQ_weak_MKO_WISE_IRAC']


In [3]:
# Compare two CEQ models 

filenames = glob.glob(os.path.join(base_dir, models[0], 'iso_age*.txt'))
filenames = [os.path.split(fn)[-1] for fn in filenames]

for fn in filenames:

    fn1 = os.path.join(base_dir, models[0], fn)
    fn2 = os.path.join(base_dir, models[1], fn)

    f1 = np.loadtxt(fn1)
    f2 = np.loadtxt(fn2)

    # columns 0 - 4: Mass, Teff, logL, R, logg
    for i in range(5):
        if all(f1[:,i] == f2[:,i]) == False:
            print(f'file {fn} columns {i} not identical')

# Models isochrones_CEQ_MKO_WISE_IRAC and isochrones_CEQ_NIRCAM_modA 
# have same physical columns


In [4]:

def parse_age(filename):
    """Extract age from a file such as iso_age0.10.txt."""
    prefix = "iso_age"
    suffix = ".txt"

    if not filename.startswith(prefix) or not filename.endswith(suffix):
        raise ValueError(f"Unexpected age filename: {filename}")

    return float(filename[len(prefix):-len(suffix)])

def format_number(value):
    """Format a grid value consistently for HDF5 names."""
    return f"{value:.3f}"


# ------------------------------------------------------------
# HDF5 conversion
# ------------------------------------------------------------

def convert_grid_models(models, input_dir, output_dir):

    astro_columns = ['Mini','logT','logL','R','logg']
    phot1_columns = ['MKO_Y', 'MKO_J', 'MKO_H', 'MKO_K', 'MKO_Lp', 'MKO_Mp', 
                     'W1', 'W2', 'W3', 'W4', 'IRAC_CH1', 'IRAC_CH2']
    phot2_columns = ['F480M', 'F187N', 'F356W', 'F150W2', 'F210M', 'F470N', 'F150W', 
                     'F405N', 'F335M', 'F140M', 'F182M', 'F070W', 'F200W', 'F360M', 
                     'F460M', 'F466N','F323N', 'F430M', 'F212N', 'F410M', 'F300M',
                     'F444W', 'F250M', 'F277W','F162M', 'F090W', 'F164N', 'F322W2', 'F115W']

    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    feh = 0.0
    output_file = output_dir / f"iso_feh{feh}.h5"

    age_files1 = sorted(glob.glob(os.path.join(input_dir, models[0], 'iso_age*.txt')))
    age_files2 = sorted(glob.glob(os.path.join(input_dir, models[1], 'iso_age*.txt')))
    age_files3 = sorted(glob.glob(os.path.join(input_dir, models[2], 'iso_age*.txt')))

    with h5py.File(output_file, "a") as h5:
        h5.attrs["model"] = 'ATMO'
        h5.attrs["FeH"] = feh

        ceq_group = h5.create_group('CEQ')

        for txt_file1, txt_file2 in zip(age_files1,age_files2):
            age = parse_age(os.path.split(txt_file1)[-1])

            age_group = ceq_group.create_group(f"age_{format_number(age)}")

            data1 = np.loadtxt(txt_file1)
            data2 = np.loadtxt(txt_file2)

            astro_data = data1[:,:5] # columns 1-4
            astro_data[:,1] = np.round(np.log10(astro_data[:,1]),6) # Teff to logT
            phot1_data = data1[:,5:] # MKO_WISE_IRAC
            phot2_data = data2[:,5:] # NIRCAM_modA

            astro = age_group.create_dataset(
                "Astro", data=astro_data, compression="gzip"
            )
            phot1 = age_group.create_dataset(
                "MKO_WISE_IRAC", data=phot1_data, compression="gzip"
            )
            phot2 = age_group.create_dataset(
                "NIRCAM_modA", data=phot2_data, compression="gzip"
                        )

            astro.attrs["columns"] = np.asarray(
                astro_columns, dtype=h5py.string_dtype("utf-8")
            )
            phot1.attrs["columns"] = np.asarray(
                phot1_columns, dtype=h5py.string_dtype("utf-8")
            )
            phot2.attrs["columns"] = np.asarray(
                            phot2_columns, dtype=h5py.string_dtype("utf-8")
                        )

            print(
                f"[{'CEQ'}] FeH={feh:.3f}, age={age:.3f}: "
                f"{len(data1)} rows"
            )

        neq_group = h5.create_group('NEQ_weak')

        for txt_file in age_files3:
            age = parse_age(os.path.split(txt_file)[-1])

            age_group = neq_group.create_group(f"age_{format_number(age)}")

            data = np.loadtxt(txt_file)

            astro_data = data[:,:5] # columns 1-4
            astro_data[:,1] = np.round(np.log10(astro_data[:,1]),6) # Teff to logT
            phot_data = data1[:,5:] # MKO_WISE_IRAC

            astro = age_group.create_dataset(
                            "Astro", data=astro_data, compression="gzip"
                        )
            phot = age_group.create_dataset(
                            "MKO_WISE_IRAC", data=phot_data, compression="gzip"
                        )
            astro.attrs["columns"] = np.asarray(
                            astro_columns, dtype=h5py.string_dtype("utf-8")
                        )
            phot.attrs["columns"] = np.asarray(
                            phot1_columns, dtype=h5py.string_dtype("utf-8")
                        )
            print(
                    f"[{'NEQ_weak'}] FeH={feh:.3f}, age={age:.3f}: "
                    f"{len(data)} rows"
                    )
            
        print(f"Saved {output_file}")
    

In [5]:
dir_out = 'ATMO_models'

convert_grid_models(models, base_dir, dir_out)

[CEQ] FeH=0.000, age=0.025: 90 rows
[CEQ] FeH=0.000, age=0.050: 90 rows
[CEQ] FeH=0.000, age=0.075: 90 rows
[CEQ] FeH=0.000, age=0.100: 90 rows
[CEQ] FeH=0.000, age=0.125: 90 rows
[CEQ] FeH=0.000, age=0.150: 90 rows
[CEQ] FeH=0.000, age=0.175: 90 rows
[CEQ] FeH=0.000, age=0.200: 90 rows
[CEQ] FeH=0.000, age=0.225: 90 rows
[CEQ] FeH=0.000, age=0.250: 90 rows
[CEQ] FeH=0.000, age=0.275: 90 rows
[CEQ] FeH=0.000, age=0.300: 89 rows
[CEQ] FeH=0.000, age=0.325: 89 rows
[CEQ] FeH=0.000, age=0.350: 89 rows
[CEQ] FeH=0.000, age=0.375: 89 rows
[CEQ] FeH=0.000, age=0.400: 89 rows
[CEQ] FeH=0.000, age=0.425: 89 rows
[CEQ] FeH=0.000, age=0.450: 89 rows
[CEQ] FeH=0.000, age=0.475: 89 rows
[CEQ] FeH=0.000, age=0.500: 89 rows
[CEQ] FeH=0.000, age=0.525: 89 rows
[CEQ] FeH=0.000, age=0.550: 89 rows
[CEQ] FeH=0.000, age=0.575: 89 rows
[CEQ] FeH=0.000, age=0.600: 89 rows
[CEQ] FeH=0.000, age=0.625: 89 rows
[CEQ] FeH=0.000, age=0.650: 89 rows
[CEQ] FeH=0.000, age=0.675: 89 rows
[CEQ] FeH=0.000, age=0.700: 

In [6]:
f1 = h5py.File(os.path.join(dir_out,'iso_feh0.0.h5'), 'r')
f1.visit(lambda x: print (x))

CEQ
CEQ/age_0.025
CEQ/age_0.025/Astro
CEQ/age_0.025/MKO_WISE_IRAC
CEQ/age_0.025/NIRCAM_modA
CEQ/age_0.050
CEQ/age_0.050/Astro
CEQ/age_0.050/MKO_WISE_IRAC
CEQ/age_0.050/NIRCAM_modA
CEQ/age_0.075
CEQ/age_0.075/Astro
CEQ/age_0.075/MKO_WISE_IRAC
CEQ/age_0.075/NIRCAM_modA
CEQ/age_0.100
CEQ/age_0.100/Astro
CEQ/age_0.100/MKO_WISE_IRAC
CEQ/age_0.100/NIRCAM_modA
CEQ/age_0.125
CEQ/age_0.125/Astro
CEQ/age_0.125/MKO_WISE_IRAC
CEQ/age_0.125/NIRCAM_modA
CEQ/age_0.150
CEQ/age_0.150/Astro
CEQ/age_0.150/MKO_WISE_IRAC
CEQ/age_0.150/NIRCAM_modA
CEQ/age_0.175
CEQ/age_0.175/Astro
CEQ/age_0.175/MKO_WISE_IRAC
CEQ/age_0.175/NIRCAM_modA
CEQ/age_0.200
CEQ/age_0.200/Astro
CEQ/age_0.200/MKO_WISE_IRAC
CEQ/age_0.200/NIRCAM_modA
CEQ/age_0.225
CEQ/age_0.225/Astro
CEQ/age_0.225/MKO_WISE_IRAC
CEQ/age_0.225/NIRCAM_modA
CEQ/age_0.250
CEQ/age_0.250/Astro
CEQ/age_0.250/MKO_WISE_IRAC
CEQ/age_0.250/NIRCAM_modA
CEQ/age_0.275
CEQ/age_0.275/Astro
CEQ/age_0.275/MKO_WISE_IRAC
CEQ/age_0.275/NIRCAM_modA
CEQ/age_0.300
CEQ/age_0.300/